# Baseline models — ACSIncome (Illinois, 2018)

Logistic Regression and Random Forest establish the validation-set reference for subsequent bias-mitigation experiments. `SEX` is excluded from predictors and retained for group comparisons. The test set is reserved for final evaluation.

## Data and models

Use the shared 60/20/20 stratified split and preprocessing pipelines. Both models are fitted only on the training split.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from fairlearn.metrics import false_positive_rate, true_positive_rate
from sklearn.metrics import accuracy_score, balanced_accuracy_score

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_acs_income
from src.experiment import prepare_data_splits
from src.models import create_logistic_regression, create_random_forest

X, y, A = load_acs_income()
X_train, X_val, X_test, y_train, y_val, y_test, A_train, A_val, A_test = (
    prepare_data_splits(X, y, A)
)

models = {
    "Logistic Regression": create_logistic_regression(),
    "Random Forest": create_random_forest(),
}

for model in models.values():
    model.fit(X_train, y_train)

## Validation results

Accuracy and balanced accuracy measure predictive performance. Fairness gaps are absolute differences between the recorded male and female groups: equal opportunity difference (EOD, TPR gap), demographic parity difference (DPD, positive prediction rate gap), and FPR gap. Signed gaps are also reported to show direction.

In [2]:
def evaluate_baseline(name, model, X_eval, y_eval, A_eval):
    predictions = model.predict(X_eval)
    male = np.asarray(A_eval) == 1
    female = np.asarray(A_eval) == 2

    tpr_male = true_positive_rate(y_eval[male], predictions[male])
    tpr_female = true_positive_rate(y_eval[female], predictions[female])
    fpr_male = false_positive_rate(y_eval[male], predictions[male])
    fpr_female = false_positive_rate(y_eval[female], predictions[female])
    positive_male = np.mean(predictions[male])
    positive_female = np.mean(predictions[female])

    return {
        "Model": name,
        "Strategy": "Baseline",
        "Accuracy": accuracy_score(y_eval, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_eval, predictions),
        "EOD": abs(tpr_male - tpr_female),
        "DPD": abs(positive_male - positive_female),
        "FPR Gap": abs(fpr_male - fpr_female),
        "TPR Male": tpr_male,
        "TPR Female": tpr_female,
        "TPR Gap (M-F)": tpr_male - tpr_female,
        "Positive Rate Male": positive_male,
        "Positive Rate Female": positive_female,
        "Positive Rate Gap (M-F)": positive_male - positive_female,
        "FPR Male": fpr_male,
        "FPR Female": fpr_female,
        "FPR Gap (M-F)": fpr_male - fpr_female,
    }


baseline_results = pd.DataFrame([
    evaluate_baseline(name, model, X_val, y_val, A_val)
    for name, model in models.items()
])

summary_columns = [
    "Model", "Strategy", "Accuracy", "Balanced Accuracy", "EOD", "DPD", "FPR Gap"
]
display(baseline_results[summary_columns].round(4))

,Model,Strategy,Accuracy,Balanced Accuracy,EOD,DPD,FPR Gap
0,Logistic Regression,Baseline,0.8020,0.7901,0.0215,0.1405,0.0515
1,Random Forest,Baseline,0.7868,0.7706,0.0294,0.1305,0.0413


## Group-level rates

Signed gaps are male minus female; their sign identifies which group has the higher rate.

In [3]:
group_columns = [
    "Model", "TPR Male", "TPR Female", "TPR Gap (M-F)",
    "Positive Rate Male", "Positive Rate Female", "Positive Rate Gap (M-F)",
    "FPR Male", "FPR Female", "FPR Gap (M-F)",
]
display(baseline_results[group_columns].round(4))

,Model,TPR Male,TPR Female,TPR Gap (M-F),Positive Rate Male,Positive Rate Female,Positive Rate Gap (M-F),FPR Male,FPR Female,FPR Gap (M-F)
0,Logistic Regression,0.7422,0.7206,0.0215,0.4501,0.3096,0.1405,0.1826,0.1311,0.0515
1,Random Forest,0.7051,0.6757,0.0294,0.4289,0.2984,0.1305,0.1760,0.1346,0.0413


## Save validation results

Keep the existing CSV filenames and columns for downstream notebooks.

In [4]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

baseline_results[summary_columns].to_csv(
    results_dir / "baseline_comparison.csv", index=False
)
baseline_results.loc[
    baseline_results["Model"] == "Logistic Regression", summary_columns
].to_csv(results_dir / "logistic_regression_baseline.csv", index=False)
baseline_results.to_csv(results_dir / "baseline_group_details.csv", index=False)

## Interpretation and limitations

These are validation results, not final test estimates. No hyperparameter tuning is performed. Group disparities are descriptive and do not establish discriminatory causation. The ACS `SEX` variable uses two recorded categories and does not represent the full range of gender identities. The next experiment evaluates reweighing against these baselines.